# Project 01 Analysis — Large-Scale SNN

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/projects/project_01_large_scale_snn/analysis.ipynb)

Run after `./run.sh` to analyse simulation outputs.

In [ ]:
!nvidia-smi
!nvcc -O2 -o large_snn large_snn.cu -lm

In [ ]:
# Run baseline (no STDP)
!./large_snn 10000 2000 0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Load spikes
spikes = np.loadtxt('project01_spikes.txt')
N = 10000; T_ms = 2000.0

if len(spikes) > 0:
    neuron_ids = spikes[:, 0].astype(int)
    spike_times = spikes[:, 1]
    print(f"Spikes: {len(spikes):,}")
    print(f"Mean firing rate: {len(spikes)/(N*T_ms/1000):.1f} Hz")

    fig = plt.figure(figsize=(15, 10))
    gs  = fig.add_gridspec(3, 2)

    # Raster (sample 500 neurons)
    ax1 = fig.add_subplot(gs[0, :])
    sample_mask = neuron_ids < 500
    ax1.scatter(spike_times[sample_mask], neuron_ids[sample_mask],
                s=0.5, c=np.where(neuron_ids[sample_mask]<8000, 'steelblue', 'coral'),
                alpha=0.6, rasterized=True)
    ax1.set_xlim(0, T_ms)
    ax1.set_ylim(-1, 500)
    ax1.set_ylabel('Neuron ID')
    ax1.set_title('Raster Plot (first 500 neurons; blue=E, red=I)')

    # Population firing rate
    ax2 = fig.add_subplot(gs[1, :])
    bins = np.arange(0, T_ms+20, 20)
    counts, _ = np.histogram(spike_times, bins=bins)
    ax2.plot(bins[:-1]+10, counts/(N*20e-3), 'b-', lw=1.5)
    ax2.set_ylabel('Pop. rate (Hz)'); ax2.set_xlabel('Time (ms)')
    ax2.set_xlim(0, T_ms); ax2.grid(True, alpha=0.3)

    # ISI distribution
    ax3 = fig.add_subplot(gs[2, 0])
    isi_list = []
    for nid in np.unique(neuron_ids)[:200]:
        st = np.sort(spike_times[neuron_ids == nid])
        if len(st) > 1: isi_list.extend(np.diff(st))
    if isi_list:
        isi = np.array(isi_list)
        ax3.hist(isi, bins=50, range=(0,200), color='steelblue', alpha=0.7)
        cv = isi.std()/isi.mean()
        ax3.set_xlabel('ISI (ms)'); ax3.set_ylabel('Count')
        ax3.set_title(f'ISI Distribution (CV={cv:.2f})')
        ax3.grid(True, alpha=0.3)

    # Firing rate distribution
    ax4 = fig.add_subplot(gs[2, 1])
    rates = pd.Series(neuron_ids).value_counts().values / (T_ms/1000)
    ax4.hist(rates, bins=30, color='coral', alpha=0.7)
    ax4.set_xlabel('Firing rate (Hz)'); ax4.set_ylabel('Count')
    ax4.set_title('Per-Neuron Firing Rate Distribution')
    ax4.grid(True, alpha=0.3)

    plt.suptitle('Project 01: Baseline SNN (No STDP)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('p01_baseline.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Run with STDP
!./large_snn 10000 5000 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Weight evolution
times, means, mins, maxs = [], [], [], []
with open('project01_weights.txt') as f:
    for line in f:
        if line.startswith('#'):
            times.append(float(line.split('=')[1].split()[0]))
        else:
            vals = list(map(float, line.split()))
            means.append(vals[0]); mins.append(vals[1]); maxs.append(vals[2])

times = np.array(times[:len(means)])
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(times, means, 'b-', lw=2, label='Mean excitatory weight')
ax.fill_between(times, mins, maxs, alpha=0.2, label='Min-Max range')
ax.set_xlabel('Time (ms)', fontsize=12)
ax.set_ylabel('Synaptic weight', fontsize=12)
ax.set_title('Excitatory Weight Evolution with STDP', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('p01_stdp_weights.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scaling benchmark
import subprocess
import numpy as np
import matplotlib.pyplot as plt

N_vals = [5000, 10000, 20000, 50000]
throughputs = []

for N in N_vals:
    result = subprocess.run(['./large_snn', str(N), '500', '0'],
                            capture_output=True, text=True)
    for line in result.stdout.split('\n'):
        if 'Throughput' in line:
            throughputs.append(float(line.split()[1]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(N_vals, throughputs, 'b-o', lw=2, markersize=8)
ax.set_xlabel('Network size N', fontsize=12)
ax.set_ylabel('M neuron-steps/s', fontsize=12)
ax.set_title('Simulation Throughput vs Network Size', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for N, tp in zip(N_vals, throughputs):
    print(f"N={N:6d}: {tp:.0f} M neuron-steps/s")